# Day 2 — 보고서 전략에 따른 피처 엔지니어링

Day 1 EDA의 모델 설계 전략을 기준으로 초기 100사이클의 총수명 회귀 입력을 준비합니다. 분석 단위는 **배터리 셀 하나당 한 행**입니다.  
ΔQ 기본 입력과 A~E를 유지하고, 추가 추출·대표 변수 교체 후보를 같은 Batch 1 그룹 CV에서 비교합니다.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

## 1. 원본 로딩과 초기 관측 구간

기본 피처는 Day 1과 같은 함수로 계산합니다. 요약값은 2~100사이클, 강건 기울기는 10~100사이클, ΔQ는 Q100(V)−Q10(V), 전류는 cycle 10을 사용합니다. 확인된 빈 기록과 계산 불가 값은 결측으로 처리합니다.

In [2]:
from src.preprocess import load
from src.features import (auxiliary_features, feature_manifest, FEATURE_GROUPS,
                          SEARCH_GROUPS, FORBIDDEN_FEATURES)
cells, summary, _, _ = load()
features = auxiliary_features(cells, summary)
assert features.cell_id.is_unique
assert features.early_rows.eq(99).all()
assert features.delta_valid_fraction.eq(1).all()
display(features.groupby('batch').agg(total=('cell_id', 'size'),
    labeled=('cycle_life', 'count'), complete_initial=('early_rows', 'min')))

Reading Batch 1: 2017-05-12_batchdata_updated_struct_errorcorrect.mat


Batch 1 loaded


Reading Batch 2: 2018-02-20_batchdata_updated_struct_errorcorrect.mat


Batch 2 loaded


Reading Batch 3: 2018-04-12_batchdata_updated_struct_errorcorrect.mat


Batch 3 loaded


,total,labeled,complete_initial
batch,,,
Batch 1,46,46,99
Batch 2,47,39,99
Batch 3,46,44,99


## 2. EDA 근거와 입력 피처

ΔQ는 배치마다 같은 방향의 수명 상관을 보였습니다. 용량은 수준과 강건 추세를 구분하고, 온도는 중복을 피하려고 한 개씩 비교합니다. 정책은 첫·둘째 속도와 전환 조건, 실제 전류는 시간 가중 통계를 사용합니다.  
결측 대체·표준화는 여기서 하지 않고 학습 폴드의 Pipeline에서 적합합니다.

In [3]:
manifest = feature_manifest()
display(manifest)
display(pd.DataFrame([{'bundle': key, 'features': ', '.join(value),
    'role': '보고서 A~E' if key in FEATURE_GROUPS else '추가·교체 후보'}
    for key, value in SEARCH_GROUPS.items()]))

,feature,source,cycles,rationale
0,delta_log_var,Qdlin / Vdlin,"10, 100",전압별 용량 변화의 대표 통계
1,QD_median,summary,2–100,초기 용량 수준·변화 또는 대표 온도
2,QD_slope,summary,10–100,초기 용량 수준·변화 또는 대표 온도
3,Tavg_mean,summary,2–100,초기 용량 수준·변화 또는 대표 온도
4,rate1_C,policy_readable,실험 시작 시,충전 단계·전환 조건·부가 표기
5,switch_pct,policy_readable,실험 시작 시,충전 단계·전환 조건·부가 표기
6,rate2_C,policy_readable,실험 시작 시,충전 단계·전환 조건·부가 표기
7,newstructure,policy_readable,실험 시작 시,충전 단계·전환 조건·부가 표기
8,charge_I_mean,I / t,10,시간 가중 실측 충전 전류
9,charge_I_std,I / t,10,시간 가중 실측 충전 전류


,bundle,features,role
0,A,delta_log_var,보고서 A~E
1,B,"delta_log_var, QD_median, QD_slope",보고서 A~E
2,C,"delta_log_var, Tavg_mean",보고서 A~E
3,D,"delta_log_var, rate1_C, switch_pct, rate2_C, n...",보고서 A~E
4,E,"delta_log_var, charge_I_mean, charge_I_std, ch...",보고서 A~E
5,B_QD_change,"delta_log_var, QD_median, QD_slope, QD_change",추가·교체 후보
6,B_QD_relative,"delta_log_var, QD_median, QD_slope, QD_relativ...",추가·교체 후보
7,C_Tmax,"delta_log_var, Tmax_mean",추가·교체 후보
8,D_interaction,"delta_log_var, rate1_C, switch_pct, rate2_C, n...",추가·교체 후보
9,A_delta_min,delta_min,추가·교체 후보


## 3. 추가 추출과 처리 가정

- `QD_change`: 요약 QD100−QD10. 전압별 ΔQ(V)와 다른 통계입니다.
- `QD_relative_change`: (QD100−QD10)/QD10. 양의 유효 QD10에서만 계산합니다.
- `policy_interaction`: C1×전환 비율/100. 보고서의 정책 결합항입니다.
- `IR_nonzero_mean`: IR의 0을 미측정으로 가정한 보조 피처입니다. 원본 IR을 수정하지 않습니다.
- `IR_all_zero`: 초기 IR이 모두 0인지 기록합니다. IR과 충전시간은 주 후보에서 보류하고 후속 민감도 실험으로 비교합니다.

기본 묶음 B에 두 용량 변화를 각각 추가하고, 평균온도와 최고온도를 교체하며, ΔQ 로그 분산과 최솟값·평균을 각각 대표 입력으로 비교합니다.

In [4]:
display(features[['cell_id', 'batch', 'QD_change', 'QD_relative_change',
                  'policy_interaction', 'IR_mean', 'IR_nonzero_mean', 'IR_all_zero']].head())
input_columns = sorted({name for columns in SEARCH_GROUPS.values() for name in columns})
assert not FORBIDDEN_FEATURES.intersection(input_columns)
assert set(input_columns).issubset(features.columns)
assert not np.isinf(features[input_columns].to_numpy(dtype=float)).any()

,cell_id,batch,QD_change,QD_relative_change,policy_interaction,IR_mean,IR_nonzero_mean,IR_all_zero
0,B1_000,Batch 1,0.001375,0.001280,2.88,0.016643,0.016643,0
1,B1_001,Batch 1,0.000853,0.000790,2.88,0.016751,0.016922,0
2,B1_002,Batch 1,0.001159,0.001069,2.88,0.016600,0.016769,0
3,B1_003,Batch 1,0.000442,0.000408,3.20,0.016146,0.016311,0
4,B1_004,Batch 1,0.000692,0.000640,3.20,0.016501,0.016669,0


## 4. 결측과 학습 불가능한 상수 피처 점검

라벨 결측 셀은 지도학습에서 제외하며 기록 길이로 수명을 대신하지 않습니다. Batch 1에서 `newstructure`는 모두 0이므로 이 표기의 효과를 학습할 수 없습니다. 정책 묶음에 기록하되, 배치·집단 해석에 사용하는 한계를 명시합니다. 전체 기록 길이·종료 용량·Knee·후기 기울기는 모델 입력에서 제외합니다.

In [5]:
labeled = features[features.cycle_life.notna()]
batch1 = labeled[labeled.batch.eq('Batch 1')]
quality = pd.DataFrame({'feature': input_columns,
    'Batch1_missing': [batch1[c].isna().sum() for c in input_columns],
    'Batch1_unique': [batch1[c].nunique() for c in input_columns],
    'Batch1_constant': [batch1[c].nunique() <= 1 for c in input_columns]})
display(quality)
display(labeled.groupby('batch').newstructure.agg(['min', 'max', 'sum', 'count']))

,feature,Batch1_missing,Batch1_unique,Batch1_constant
0,QD_change,0,46,False
1,QD_median,0,46,False
2,QD_relative_change,0,46,False
3,QD_slope,0,46,False
4,Tavg_mean,0,46,False
5,Tmax_mean,0,46,False
6,charge_I_mean,0,46,False
7,charge_I_p95,0,46,False
8,charge_I_std,0,46,False
9,delta_log_var,0,46,False


,min,max,sum,count
batch,,,,
Batch 1,0,0,0,46
Batch 2,0,1,9,39
Batch 3,1,1,44,44


## 5. 저장과 재현

`features.csv`에는 라벨과 EDA 점검용 열도 함께 저장합니다. 학습 X는 `SEARCH_GROUPS`의 명시적 입력 목록으로만 선택합니다.  
`feature_manifest.csv`는 입력의 출처와 관측 시점을, `feature_quality.csv`는 Batch 1 결측·상수 여부를 기록합니다.  
`03_modeling.ipynb`는 이 파일을 읽어 모델을 선택합니다.

In [6]:
(ROOT / 'results').mkdir(exist_ok=True)
features.to_csv(ROOT / 'results/features.csv', index=False)
manifest.to_csv(ROOT / 'results/feature_manifest.csv', index=False)
quality.to_csv(ROOT / 'results/feature_quality.csv', index=False)
print(f"Saved {len(features)} cells, {len(labeled)} labeled cells; {len(SEARCH_GROUPS)} feature bundles")

Saved 139 cells, 129 labeled cells; 11 feature bundles
